# Config (Postgres + Ollama)

In [1]:
import os, re, json, uuid
from pathlib import Path
import requests
from tqdm import tqdm

# ===== Ollama (Phi-4)
OLLAMA_BASE = "http://localhost:11434"
OLLAMA_MODEL = "phi4"

def ollama_chat(messages, model=OLLAMA_MODEL, temperature=0.2, stream=False):
    url = f"{OLLAMA_BASE}/api/chat"
    payload = {
        "model": model,
        "messages": messages,
        "options": {"temperature": float(temperature)},
        "stream": bool(stream),
    }
    r = requests.post(url, json=payload, timeout=600)
    r.raise_for_status()
    return r.json()["message"]["content"]

# ===== Postgres (ajuste para o seu ambiente)
PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB", "professor_in_pocket")
PG_USER = os.getenv("PG_USER", "postgres")
PG_PASS = os.getenv("PG_PASS", "postgres")

# Dimensão do embedding (bge-m3 = 1024)
EMB_DIM = 1024

# Conexão + criação do schema pgvector

In [ ]:
import psycopg
from psycopg.rows import dict_row

def pg_connect():
    conn = psycopg.connect(
        host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASS,
        autocommit=True
    )
    return conn

DDL = f"""
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS pip_chunks (
    id UUID PRIMARY KEY,
    disciplina TEXT NOT NULL,
    source TEXT NOT NULL,
    chunk_index INT NOT NULL,
    content TEXT NOT NULL,
    embedding VECTOR({EMB_DIM}) NOT NULL
);

-- Índice IVFFlat para COSINE (recomendado criar após inserir dados)
-- CREATE INDEX IF NOT EXISTS pip_chunks_emb_ivfflat
-- ON pip_chunks USING ivfflat (embedding vector_cosine_ops) WITH (lists = 100);
"""

with pg_connect() as conn:
    with conn.cursor() as cur:
        cur.execute(DDL)

print("OK: extensão/tabela prontas.")

# Leitura de arquivos (igual ao anterior)

In [ ]:
from docx import Document as DocxDocument

def clean_text(s: str) -> str:
    s = s.replace("\u00a0", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def read_txt(path: Path) -> str:
    return clean_text(path.read_text(encoding="utf-8", errors="ignore"))

def read_docx(path: Path) -> str:
    doc = DocxDocument(str(path))
    parts = [p.text for p in doc.paragraphs if p.text.strip()]
    return clean_text("\n".join(parts))

def read_pdf(path: Path) -> str:
    from pypdf import PdfReader
    reader = PdfReader(str(path))
    pages = []
    for pg in reader.pages:
        t = pg.extract_text() or ""
        if t.strip():
            pages.append(t)
    return clean_text("\n".join(pages))

def load_documents(folder: str):
    folder = Path(folder)
    assert folder.exists(), f"Pasta não encontrada: {folder}"
    docs = []
    for p in folder.rglob("*"):
        if p.is_dir():
            continue
        ext = p.suffix.lower()
        try:
            if ext in [".txt", ".md"]:
                text = read_txt(p)
            elif ext == ".docx":
                text = read_docx(p)
            elif ext == ".pdf":
                text = read_pdf(p)
            else:
                continue
            if text.strip():
                docs.append({"path": str(p), "text": text})
        except Exception as e:
            print("Falha ao ler", p, "->", e)
    return docs

DISCIPLINA_DIR = r"./data/disciplinas/eng_comp_programacao"
docs = load_documents(DISCIPLINA_DIR)
len(docs), [d["path"] for d in docs[:3]]

# Chunking

In [ ]:
def chunk_text(text: str, chunk_size=900, overlap=150):
    text = clean_text(text)
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks

all_chunks = []
for d in docs:
    chs = chunk_text(d["text"], chunk_size=900, overlap=150)
    for i, ch in enumerate(chs):
        all_chunks.append({
            "id": str(uuid.uuid4()),
            "disciplina": "programacao",
            "source": d["path"],
            "chunk_index": i,
            "content": ch,
        })

len(all_chunks), all_chunks[0]["source"]

# Embeddings (BGE-M3) e INSERT no Postgres

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-m3")  # https://huggingface.co/BAAI/bge-m3

def embed_texts(texts, batch_size=16):
    vecs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        v = embedder.encode(batch, normalize_embeddings=True)  # bom para cosine
        vecs.extend(v.tolist())
    return vecs

# Embeda tudo
texts = [c["content"] for c in all_chunks]
vecs = embed_texts(texts, batch_size=16)

# Insere (UPSERT)
UPSERT = """
INSERT INTO pip_chunks (id, disciplina, source, chunk_index, content, embedding)
VALUES (%s::uuid, %s, %s, %s, %s, %s::vector)
ON CONFLICT (id) DO UPDATE SET
  disciplina = EXCLUDED.disciplina,
  source = EXCLUDED.source,
  chunk_index = EXCLUDED.chunk_index,
  content = EXCLUDED.content,
  embedding = EXCLUDED.embedding;
"""

with pg_connect() as conn:
    with conn.cursor() as cur:
        for c, v in tqdm(list(zip(all_chunks, vecs)), total=len(all_chunks)):
            cur.execute(UPSERT, (c["id"], c["disciplina"], c["source"], c["chunk_index"], c["content"], v))

print("OK: chunks indexados no Postgres.")

# Criar índice IVFFlat (cosine) + probes

In [ ]:
LISTS = 100  # comece assim; depois você ajusta pelo volume de dados
CREATE_INDEX = f"""
CREATE INDEX IF NOT EXISTS pip_chunks_emb_ivfflat
ON pip_chunks USING ivfflat (embedding vector_cosine_ops) WITH (lists = {LISTS});
"""

with pg_connect() as conn:
    with conn.cursor() as cur:
        cur.execute(CREATE_INDEX)

print("OK: índice IVFFlat (cosine) criado.")

# Recuperação top-k usando pgvector (<=> cosine distance)

In [ ]:
def retrieve_pgvector(query: str, k=5, disciplina="programacao", probes=10):
    q_emb = embedder.encode([query], normalize_embeddings=True)[0].tolist()

    SQL = f"""
    SET ivfflat.probes = {int(probes)};

    SELECT
      source,
      chunk_index,
      content,
      (embedding <=> %s::vector) AS distance
    FROM pip_chunks
    WHERE disciplina = %s
    ORDER BY embedding <=> %s::vector
    LIMIT {int(k)};
    """

    with pg_connect() as conn:
        with conn.cursor(row_factory=dict_row) as cur:
            cur.execute(SQL, (q_emb, disciplina, q_emb))
            rows = cur.fetchall()

    hits = []
    for r in rows:
        hits.append({
            "source": r["source"],
            "chunk_index": r["chunk_index"],
            "text": r["content"],
            "distance": float(r["distance"]),
        })
    return hits

def format_sources(hits, max_chars_each=450):
    lines = []
    for i, h in enumerate(hits, 1):
        excerpt = h["text"][:max_chars_each].replace("\n", " ").strip()
        lines.append(f"[{i}] {h['source']} (chunk {h['chunk_index']})\n    Trecho: {excerpt}...")
    return "\n".join(lines)

test_hits = retrieve_pgvector("Qual é o objetivo do Professor in Pocket?", k=4, probes=10)
print(format_sources(test_hits))

# Resposta com Phi-4 (RAG + fontes)

In [ ]:
SYSTEM_RULES = """Você é um tutor acadêmico.
Regras obrigatórias:
1) Use SOMENTE o contexto fornecido (material oficial).
2) Se a resposta não estiver no contexto, diga claramente: "Não encontrei base suficiente no material fornecido."
3) Sempre inclua uma seção "Fontes" citando [1], [2], etc.
4) Seja direto e didático, sem inventar detalhes.
"""

def answer_rag_pgvector(query: str, k=5, probes=10, disciplina="programacao"):
    hits = retrieve_pgvector(query, k=k, probes=probes, disciplina=disciplina)
    context = "\n\n".join([f"[{i+1}] {h['text']}" for i, h in enumerate(hits)])
    sources = format_sources(hits)

    user_prompt = f"""Pergunta: {query}

Contexto (trechos do material):
{context}

Agora responda seguindo as regras. No final, traga "Fontes" referenciando os índices [1], [2]...
"""
    messages = [
        {"role": "system", "content": SYSTEM_RULES},
        {"role": "user", "content": user_prompt},
    ]
    resp = ollama_chat(messages, temperature=0.2)
    resp += "\n\n---\nFontes (auditável):\n" + sources
    return resp

print(answer_rag_pgvector("O que é o Professor in Pocket e qual o objetivo central?", k=5, probes=10)[:2500])

# Testes de validação

In [ ]:
tests = [
    "O que é o Professor in Pocket e qual o objetivo central?",
    "Como funciona a camada de orquestração e o classificador?",
    "Quais são as fases do piloto e o cronograma estimado?",
    "Qual é a cor favorita do reitor? (isso não está no material)",
]

for q in tests:
    print("\n" + "="*90)
    print("Q:", q)
    print(answer_rag_pgvector(q, k=5, probes=10)[:1800])